In [1]:
from pathlib import Path
from lamf_analysis.code_ocean import capsule_data_utils as cdu

import register_fov_local_zstack as reg_mod
import register_fov_local_zstack_qc as qc_mod
import register_fov_local_zstack_viz as viz_mod

import register_fov_local_zstack_parallel as par_mod

%load_ext autoreload
%autoreload 2

In [4]:
input_dir = Path("/root/capsule/data")
out_dir = Path("/root/capsule/results")
num_planes = 8
input_data = list(input_dir.glob('multiplane-ophys*'))
assert len(input_data) == 1, f"Expected exactly one input file, found {len(input_data)}"
input_folder = input_data[0]
plane_ids = cdu.get_plane_ids_from_processed_path(input_folder)
print(f"Found plane IDs: {plane_ids}")
assert len(plane_ids) == num_planes, f"Expected {num_planes} plane IDs, found {len(plane_ids)}"
plane_paths = [input_folder / plane_id for plane_id in plane_ids]

Found plane IDs: ['VISp_0', 'VISp_1', 'VISp_2', 'VISp_3', 'VISp_4', 'VISp_5', 'VISp_6', 'VISp_7']


In [7]:
def run_plane(plane_path, out_dir):
    plane_out_dir = out_dir / plane_path.name
    qc_dir = plane_out_dir / 'qc'
    qc_dir.mkdir(parents=True, exist_ok=True)

    # End-to-end registration + save
    run_out = reg_mod.run_single_plane(
        plane_path=plane_path,
        output_dir=plane_out_dir,
        reg_ref_ind=0,
        save_tiff=True,
        file_suffix='local_zstack_to_fov',
        pad=3,
    )

    result = run_out['result']
    saved_paths = run_out['saved_paths']

    # Method comparison figure
    viz_mod.save_method_comparison_figure(
        result=result,
        row_i=0,
        out_path=qc_dir / 'registration_method_comparison.png',
    )

    # QC (ROI overlay + GIF)
    qc_out = qc_mod.run_qc(result, qc_dir)
    print(saved_paths)
    print(qc_out['overlay_image_path'])
    print(qc_out['gif_path'])

In [8]:
# running a single plane for testing
run_plane(plane_paths[0], out_dir)

Error getting pixel size from the raw path: Raw path not found (/root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40)
Trying session.json
{'h5_path': PosixPath('/root/capsule/results/VISp_0/782149_2025-04-15_VISp_0_local_zstack_to_fov_results.h5'), 'metadata_path': PosixPath('/root/capsule/results/VISp_0/782149_2025-04-15_VISp_0_local_zstack_to_fov_metadata.json')}
/root/capsule/results/VISp_0/qc/782149_2025-04-15_VISp_0_registered_mean_zstack_roi_overlay.png
/root/capsule/results/VISp_0/qc/782149_2025-04-15_VISp_0_registered_zstack_with_roi_outlines.gif


In [10]:
# running all planes in a loop
for plane_path in plane_paths[1:]:
    run_plane(plane_path, out_dir)
# took 8 minutes (about 1 minute per plane)

Error getting pixel size from the raw path: Raw path not found (/root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40)
Trying session.json
{'h5_path': PosixPath('/root/capsule/results/VISp_1/782149_2025-04-15_VISp_1_local_zstack_to_fov_results.h5'), 'metadata_path': PosixPath('/root/capsule/results/VISp_1/782149_2025-04-15_VISp_1_local_zstack_to_fov_metadata.json')}
/root/capsule/results/VISp_1/qc/782149_2025-04-15_VISp_1_registered_mean_zstack_roi_overlay.png
/root/capsule/results/VISp_1/qc/782149_2025-04-15_VISp_1_registered_zstack_with_roi_outlines.gif
Error getting pixel size from the raw path: Raw path not found (/root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40)
Trying session.json
{'h5_path': PosixPath('/root/capsule/results/VISp_2/782149_2025-04-15_VISp_2_local_zstack_to_fov_results.h5'), 'metadata_path': PosixPath('/root/capsule/results/VISp_2/782149_2025-04-15_VISp_2_local_zstack_to_fov_metadata.json')}
/root/capsule/results/VISp_2/qc/782149_2025-04-15_V

In [ ]:
plane_paths = [input_folder / plane_id for plane_id in plane_ids]

results = par_mod.run_planes_parallel(
    plane_paths,
    output_dir=Path("/root/capsule/results"),
    n_workers=8,
)

par_mod.print_parallel_results(results)
# About 5 minutes for 8 planes with n_workers=8
# Not much gain from parallelization.


Parallel Registration Summary
Total planes: 8
Successful: 8
Failed: 0

✓ /root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40_processed_2025-09-12_02-33-10/VISp_3
  Results: /root/capsule/results/VISp_3/782149_2025-04-15_VISp_3_local_zstack_to_fov_results.h5
  Selected: affine_after_translation

✓ /root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40_processed_2025-09-12_02-33-10/VISp_7
  Results: /root/capsule/results/VISp_7/782149_2025-04-15_VISp_7_local_zstack_to_fov_results.h5
  Selected: affine_after_translation

✓ /root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40_processed_2025-09-12_02-33-10/VISp_0
  Results: /root/capsule/results/VISp_0/782149_2025-04-15_VISp_0_local_zstack_to_fov_results.h5
  Selected: affine_after_translation

✓ /root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40_processed_2025-09-12_02-33-10/VISp_5
  Results: /root/capsule/results/VISp_5/782149_2025-04-15_VISp_5_local_zstack_to_fov_results.h5
  Selected: affine_aft

In [12]:
# load results to regenerate qc figures
for plane_id in plane_ids[:2]:
    plane_out_dir = out_dir / plane_id
    qc_dir = plane_out_dir / 'qc'

    metadata_files = sorted(plane_out_dir.glob('*_metadata.json'))
    if not metadata_files:
        print(f"[SKIP] No metadata file found in {plane_out_dir}")
    else:

        metadata_path = metadata_files[0]
        print(f"Loading {metadata_path.name} ...")
        result = viz_mod.load_result_from_bundle(metadata_path)

        qc_out = qc_mod.run_qc(result, qc_dir)
        print(f"  overlay → {qc_out['overlay_image_path']}")
        print(f"  gif     → {qc_out['gif_path']}")

Loading 782149_2025-04-15_VISp_0_local_zstack_to_fov_metadata.json ...
Error getting pixel size from the raw path: Raw path not found (/root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40)
Trying session.json
  overlay → /root/capsule/results/VISp_0/qc/782149_2025-04-15_VISp_0_registered_mean_zstack_roi_overlay.png
  gif     → /root/capsule/results/VISp_0/qc/782149_2025-04-15_VISp_0_registered_zstack_with_roi_outlines.gif
Loading 782149_2025-04-15_VISp_1_local_zstack_to_fov_metadata.json ...
Error getting pixel size from the raw path: Raw path not found (/root/capsule/data/multiplane-ophys_782149_2025-04-15_09-36-40)
Trying session.json
  overlay → /root/capsule/results/VISp_1/qc/782149_2025-04-15_VISp_1_registered_mean_zstack_roi_overlay.png
  gif     → /root/capsule/results/VISp_1/qc/782149_2025-04-15_VISp_1_registered_zstack_with_roi_outlines.gif
